# PLP Data Analytics Assignment 3

## Engagement 1: Duka Electronics Sales Exploration

**Student:** Christine Wanjira

**Project:** Duka Sales and Agent Cleanup

### Project Objective

To explore Duka Electronics sales data for January–June 2026, identify business trends, compare branch performance and provide useful recommendations to management.

## Executive Summary

The Duka Electronics dataset contains 1,000 sales orders from 5 branches between January and June 2026. Nairobi recorded the highest branch revenue of KES 26,106,100. Laptops generated the highest product revenue of KES 41,535,000. April recorded the highest monthly revenue of KES 13,518,300, while June recorded the lowest at KES 10,903,400. The filtered dataset contained 83 Premium Online orders from Nairobi and Mombasa, generating total revenue of KES 10,218,000.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
%%writefile generate_data.py

import numpy as np

import pandas as pd

rng = np.random.default_rng(2026)

n = 1000

products = {"Phone": 15000, "Laptop": 65000, "Headphones": 2500,
            "Tablet": 28000, "Charger": 1200}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n,
                         p=[0.4, 0.2, 0.15, 0.15, 0.1]),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(["In-store", "Online", "Phone order"], n,
                          p=[0.55, 0.35, 0.10]),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)

Writing generate_data.py


In [ ]:
%%writefile -a generate_data.py

import numpy as np, pandas as pd

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS")
                   .strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ",
                            "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01")
                 + pd.to_timedelta(rng.integers(0, 60, n), unit="D"))
                .strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)

txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

Appending to generate_data.py


In [ ]:
%run generate_data.py

In [ ]:
print("Duka sales:", sales.shape)
print("Agents:", agents.shape)
print("Agent transactions:", txns.shape)

Duka sales: (1000, 7)
Agents: (20, 3)
Agent transactions: (612, 5)


In [ ]:
duka = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("First 5 rows:")
display(duka.head())

print("\nDataset shape:")
print(duka.shape)

print("\nColumn data types:")
print(duka.dtypes)

print("\nMissing values:")
print(duka.isna().sum())

print("\nDate range:")
print(duka["order_date"].min(), "to", duka["order_date"].max())

print("\nBranches:")
print(duka["branch"].unique())

print("\nProducts:")
print(duka["product"].unique())

print("\nSales channels:")
print(duka["channel"].unique())

First 5 rows:


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500



Dataset shape:
(1000, 7)

Column data types:
order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

Missing values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date range:
2026-01-01 00:00:00 to 2026-06-29 00:00:00

Branches:
['Nakuru' 'Nairobi' 'Mombasa' 'Eldoret' 'Kisumu']

Products:
['Phone' 'Laptop' 'Tablet' 'Headphones' 'Charger']

Sales channels:
['In-store' 'Online' 'Phone order']


## First Five Minutes: Data Inspection

The Duka Electronics dataset contains 1,000 rows and 7 columns. The data includes order IDs, order dates, branches, products, quantities, sales channels and unit prices.

The order_date column is correctly stored as datetime, while order_id, quantity and unit_price are integers. The remaining columns are stored as text.

There are no missing values in any of the seven columns. The sales period runs from 1 January 2026 to 29 June 2026.

The dataset covers five branches: Nakuru, Nairobi, Mombasa, Eldoret and Kisumu. It contains five products: Phone, Laptop, Tablet, Headphones and Charger, sold through three channels: In-store, Online and Phone order.

The initial inspection suggests that the dataset is complete and ready for further sales analysis.

In [ ]:
duka["revenue"] = duka["quantity"] * duka["unit_price"]

duka["month"] = duka["order_date"].dt.month_name()

duka["weekday"] = duka["order_date"].dt.day_name()

duka["is_weekend"] = duka["order_date"].dt.dayofweek >= 5

duka["price_band"] = np.where(
    duka["unit_price"] >= 28000,
    "Premium",
    "Value"
)

display(duka.head())

print("Updated columns:")
print(duka.columns.tolist())

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


Updated columns:
['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel', 'unit_price', 'revenue', 'month', 'weekday', 'is_weekend', 'price_band']


In [ ]:
branch_summary = duka.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    number_of_orders=("order_id", "count")
)

branch_summary = branch_summary.sort_values(
    "total_revenue",
    ascending=False
)

display(branch_summary)

,total_revenue,number_of_orders
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117


### Branch Performance Findings

Nairobi is the best-performing branch, generating KES 26,106,100 from 376 orders. Mombasa ranks second with KES 14,003,700 from 186 orders. Eldoret has the lowest performance, generating KES 8,844,200 from 117 orders. The ranking by total revenue is the same as the ranking by number of orders. Management should maintain Nairobi's strong performance and investigate ways to improve sales in Eldoret.

In [ ]:
product_summary = duka.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum")
)

top_revenue = product_summary.sort_values(
    "total_revenue", ascending=False
).head(3)

top_units = product_summary.sort_values(
    "units_sold", ascending=False
).head(3)

print("Top 3 products by revenue:")
display(top_revenue)

print("Top 3 products by units sold:")
display(top_units)

Top 3 products by revenue:


,total_revenue,units_sold
product,,
Laptop,41535000,639
Tablet,19348000,691
Phone,8880000,592


Top 3 products by units sold:


,total_revenue,units_sold
product,,
Tablet,19348000,691
Laptop,41535000,639
Phone,8880000,592


### Product Performance Findings

Laptops generate the highest revenue at KES 41,535,000 from 639 units sold. Tablets rank second in revenue at KES 19,348,000 but lead in sales volume with 691 units. Phones rank third in both revenue, at KES 8,880,000, and units sold, at 592 units. This shows that high-priced products can generate more revenue even when fewer units are sold. Management should continue monitoring laptop profitability while maintaining tablet stock because of its high sales volume.

In [ ]:
weekend_summary = duka.groupby("is_weekend").agg(
    number_of_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
)

weekend_summary.index = [
    "Weekday" if not value else "Weekend"
    for value in weekend_summary.index
]

display(weekend_summary)

,number_of_orders,total_revenue,average_order_revenue
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


### Weekend vs Weekday Sales Findings

Weekdays recorded 723 orders and generated KES 51,856,200 in revenue. Weekends recorded 277 orders and generated KES 19,936,300. The average revenue per order was KES 71,723.65 on weekdays and KES 71,972.20 on weekends. Although weekdays generate more total revenue because they have more orders, weekend orders have a slightly higher average value. Management could encourage weekend promotions to increase the number of weekend orders.

In [ ]:
monthly_revenue = duka.groupby("month")["revenue"].sum()

month_order = [
    "January", "February", "March",
    "April", "May", "June"
]

monthly_revenue = monthly_revenue.reindex(month_order)

display(monthly_revenue)

best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

percentage_difference = (
    (monthly_revenue.max() - monthly_revenue.min())
    / monthly_revenue.min()
) * 100

print("Best month:", best_month)
print("Worst month:", worst_month)
print("Percentage difference:", round(percentage_difference, 2), "%")

,revenue
month,
January,11786600
February,11772000
March,12396800
April,13518300
May,11415400
June,10903400


Best month: April
Worst month: June
Percentage difference: 23.98 %


### Monthly Revenue Findings

April was the best-performing month, generating KES 13,518,300 in revenue. June was the worst-performing month, generating KES 10,903,400. The revenue difference between April and June was 23.98%, calculated using June revenue as the baseline. Revenue increased from January to April before declining in May and June. Management should investigate the factors behind April's strong performance and the decline towards June.

In [ ]:
import os

premium_online_coast_capital = duka[
    (duka["price_band"] == "Premium") &
    (duka["channel"] == "Online") &
    (duka["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

print("Number of matching orders:", len(premium_online_coast_capital))

print("Total revenue (KES):",
      premium_online_coast_capital["revenue"].sum())

display(premium_online_coast_capital.head())

os.makedirs("outputs", exist_ok=True)

premium_online_coast_capital.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("File saved successfully!")

Number of matching orders: 83
Total revenue (KES): 10218000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


File saved successfully!


### Premium Online Orders: Nairobi and Mombasa

The analysis identified 83 orders matching the Premium, Online and Nairobi or Mombasa conditions. These orders generated total revenue of KES 10,218,000. The filtered records were exported to outputs/premium_online_coast_capital.csv using index=False to avoid adding an unnecessary index column. This export can help management examine premium online sales in the capital and coastal branches.

In [ ]:
import os

os.makedirs("/content/excel", exist_ok=True)

print("Excel folder created!")

Excel folder created!


In [ ]:
import os
os.makedirs("/content/excel", exist_ok=True)
print("Excel folder is ready!")

Excel folder is ready!


In [ ]:
import shutil

shutil.move(
    "/content/duka_branch_summary.xlsx",
    "/content/excel/duka_branch_summary.xlsx"
)

print("Workbook moved successfully!")

Workbook moved successfully!


In [ ]:
import os

print(os.listdir("/content/excel"))

['duka_branch_summary.xlsx']


In [ ]:
import os

print(os.listdir("/content/outputs"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/outputs'

In [ ]:
import os

os.makedirs("/content/outputs", exist_ok=True)

print("Outputs folder created!")

Outputs folder created!


In [ ]:
import os

print(os.listdir("/content"))

['.config', '.ipynb_checkpoints', 'excel', 'outputs', 'sample_data']


In [ ]:
import os

print(os.listdir("/content"))

['.config', '.ipynb_checkpoints', 'excel', 'outputs', 'duka_sales.csv', 'sample_data']


In [ ]:
import pandas as pd
import numpy as np
import os

duka = pd.read_csv("/content/duka_sales.csv")

duka["revenue"] = duka["quantity"] * duka["unit_price"]

duka["price_band"] = np.where(
    duka["unit_price"] >= 28000,
    "Premium",
    "Value"
)

premium_online_coast_capital = duka[
    (duka["price_band"] == "Premium") &
    (duka["channel"] == "Online") &
    (duka["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

premium_online_coast_capital.to_csv(
    "/content/outputs/premium_online_coast_capital.csv",
    index=False
)

print("Matching orders:", len(premium_online_coast_capital))
print("Total revenue (KES):", premium_online_coast_capital["revenue"].sum())

Matching orders: 83
Total revenue (KES): 10218000


In [ ]:
import os

print(os.listdir("/content/outputs"))

['premium_online_coast_capital.csv']


In [ ]:
import os

for folder in ["/content", "/content/excel", "/content/outputs"]:
    print(f"\n{folder}")
    print(os.listdir(folder))


/content
['.config', '.ipynb_checkpoints', 'excel', 'outputs', 'duka_sales.csv', 'sample_data']

/content/excel
['duka_branch_summary.xlsx']

/content/outputs
['premium_online_coast_capital.csv']


In [ ]:
import os

print("Files in Colab:")
print(os.listdir("/content"))

Files in Colab:
['.config', '.ipynb_checkpoints', 'excel', 'outputs', 'duka_sales.csv', 'sample_data']


In [ ]:
import os

print(os.listdir("/content"))

['.config', '.ipynb_checkpoints', 'excel', 'outputs', 'generate_data.py', 'duka_sales.csv', 'sample_data']


In [14]:
import os

print(os.path.exists("/content/cleaning_pipeline.py"))

False


In [15]:
%%writefile /content/cleaning_pipeline.py

import pandas as pd
import numpy as np

def clean_agent_data(txns, agents):
    """
    Clean agent transaction data and return a cleaned DataFrame
    together with a log of the cleaning actions.
    """
    clean = txns.copy()
    log = {}
    log["original_rows"] = len(clean)

    # Remove duplicate transactions
    before = len(clean)
    clean = clean.drop_duplicates(
        subset="txn_id", keep="first"
    ).copy()
    log["duplicate_rows_removed"] = before - len(clean)

    # Standardise transaction types
    clean["txn_type"] = (
        clean["txn_type"].astype(str).str.strip().str.lower()
    )
    clean["txn_type"] = clean["txn_type"].map({
        "deposit": "Deposit",
        "withdrawal": "Withdrawal"
    })

    # Convert amounts and fill missing values
    clean["amount"] = pd.to_numeric(
        clean["amount"], errors="coerce"
    )
    clean["amount_imputed"] = clean["amount"].isna()
    median_amount = clean["amount"].median()
    clean["amount"] = clean["amount"].fillna(median_amount)

    # Flag outliers using IQR
    q1 = clean["amount"].quantile(0.25)
    q3 = clean["amount"].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    clean["is_outlier"] = (
        (clean["amount"] < lower_limit) |
        (clean["amount"] > upper_limit)
    )

    # Standardise transaction dates
    clean["txn_date"] = pd.to_datetime(
        clean["txn_date"],
        format="mixed",
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    # Add county information
    clean = clean.merge(
        agents[["agent_id", "county"]],
        on="agent_id",
        how="left"
    )

    log["unknown_agent_transactions"] = clean["county"].isna().sum()
    log["final_rows"] = len(clean)

    return clean, log

Writing /content/cleaning_pipeline.py


In [16]:
import os

print(os.path.exists("/content/cleaning_pipeline.py"))

True


In [17]:
import os
print(os.path.exists("/content/requirements.txt"))

False


In [18]:
%%writefile /content/requirements.txt

pandas
numpy

Writing /content/requirements.txt


In [19]:
import os

print(os.path.exists("/content/requirements.txt"))

True


In [20]:
import os
print(os.path.exists("/content/README.md"))

False


In [21]:
%%writefile /content/README.md

# PLP Data Analytics Assignment 3
## Duka Sales and Agent Cleanup

**Student:** Christine Wanjira

## Project Overview

This project contains two data analytics engagements:

1. Duka Electronics Sales Exploration.
2. Agent Transactions Data Cleaning and Quality Assessment.

## Engagement 1: Duka Electronics Sales Exploration

The first engagement analyses 1,000 Duka Electronics sales orders from five branches between January and June 2026.

The analysis explores branch performance, product revenue, monthly trends, weekday and weekend sales, and price categories.

### Executive Summary

The Duka Electronics dataset contains 1,000 sales orders from 5 branches between January and June 2026. Nairobi recorded the highest branch revenue of KES 26,106,100. Laptops generated the highest product revenue of KES 41,535,000. April recorded the highest monthly revenue of KES 13,518,300, while June recorded the lowest at KES 10,903,400. The filtered dataset contained 83 Premium Online orders from Nairobi and Mombasa, generating total revenue of KES 10,218,000.

### Duka Output Files

- `excel/duka_branch_summary.xlsx`
- `outputs/premium_online_coast_capital.csv`

## Engagement 2: Agent Transactions Cleanup

The second engagement focuses on cleaning agent transaction data, identifying data quality issues and preparing reliable data for analysis.

The cleaning pipeline performs the following tasks:

- Removes duplicate transaction IDs.
- Standardises transaction types.
- Converts transaction amounts to numeric values.
- Imputes missing amounts using the median.
- Flags potential outliers using the IQR method.
- Standardises transaction dates.
- Merges agent county information.

### Agent Data Before and After Cleaning

- Original transaction rows: 612
- Duplicate rows removed: 12
- Final cleaned transaction rows: 600
- Unknown agent transactions after cleaning: 7

### Agent Output File

- `outputs/agent_transactions_clean.csv`

## Project Files

- `generate_data.py` - Generates the Duka and agent datasets.
- `cleaning_pipeline.py` - Contains the agent data cleaning function.
- `duka_sales_exploration.ipynb` - Duka sales analysis.
- `agent_cleanup.ipynb` - Agent data cleaning and quality report.
- `requirements.txt` - Required Python libraries.
- `excel/` - Excel analysis workbook.
- `outputs/` - Final exported datasets.

## How to Run the Project

### 1. Install Requirements

```bash
pip install -r requirements.txt

Writing /content/README.md


In [22]:
import os

print(os.path.exists("/content/README.md"))

True


In [23]:
import os

print(os.listdir("/content"))

['.config', 'requirements.txt', '.ipynb_checkpoints', 'excel', 'cleaning_pipeline.py', 'outputs', 'generate_data.py', 'duka_sales.csv', 'README.md', 'sample_data']
